# 01 — Data collection and provenance

Source inventory and provenance only. Inspect the immutable-source registry and validate representative current archives. This notebook makes no downloads and writes no data; reusable validation lives in `src/collection_validation.py`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.collection_validation import validate_icnf_archive, validate_zip_archive
from src.config import OPERATIONAL_FORECAST
from src.source_registry import CAOP_2025, ICNF_ANNUAL_ARCHIVES, REGISTERED_SOURCES

print(f'Project root: {PROJECT_ROOT}')
print(f'Registered immutable ZIP sources: {len(REGISTERED_SOURCES)}')
print('No automated downloads are executed.')

## Read-only archive checks

Validate the CAOP archive and the ICNF years needed at the current labelled/scoring boundary. Invalid ICNF geometries remain immutable source facts and are repaired only in derived processing.

In [ ]:
print('CAOP:', validate_zip_archive(CAOP_2025, PROJECT_ROOT)['zip_integrity'])
for year in (2023, 2024, 2025):
    result = validate_icnf_archive(ICNF_ANNUAL_ARCHIVES[year], PROJECT_ROOT, expected_year=year)
    print(year, result['feature_count'], result['non_empty_geometry_count'], result['invalid_geometry_count'])

## Current annual temporal roles

For forecast year 2026, predictor inputs are from T=2025. The model is labelled only through T=2024 / observed outcome 2025. Same-year ICNF burned area is never a predictor.

In [ ]:
forecast_year = OPERATIONAL_FORECAST.current_forecast_year
predictor_year = OPERATIONAL_FORECAST.predictor_year(forecast_year)
history_years = OPERATIONAL_FORECAST.history_years(forecast_year)
assert max(history_years) < predictor_year
print({'forecast_year': forecast_year, 'predictor_year': predictor_year, 'history_years': history_years, 'latest_observed_outcome': OPERATIONAL_FORECAST.latest_observed_outcome_year(forecast_year)})